In [31]:
import mne
import openneuro
import pandas as pd
import numpy as np
np.set_printoptions(threshold=np.inf)
np.set_printoptions(linewidth=np.inf)
from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
	write_raw_bids
)

In [14]:
# From https://github.com/braindecode/braindecode/blob/master/braindecode/datasets/tuh.py
def _rename_channels(raw):
	"""
	Renames the EEG channels using mne conventions and sets their type to 'eeg'.

	See https://isip.piconepress.com/publications/reports/2020/tuh_eeg/electrodes/
	"""
	# remove ref suffix and prefix:
	# TODO: replace with removesuffix and removeprefix when 3.8 is dropped
	mapping_strip = {
		c: c.replace("-REF", "").replace("-LE", "").replace("EEG ", "")
		for c in raw.ch_names
	}
	raw.rename_channels(mapping_strip)

	montage1005 = mne.channels.make_standard_montage("standard_1005")
	mapping_eeg_names = {
		c.upper(): c for c in montage1005.ch_names if c.upper() in raw.ch_names
	}

	# Set channels whose type could not be inferred (defaulted to "eeg") to "misc":
	non_eeg_names = [c for c in raw.ch_names if c not in mapping_eeg_names]
	if non_eeg_names:
		non_eeg_types = raw.get_channel_types(picks=non_eeg_names)
		mapping_non_eeg_types = {
			c: "misc" for c, t in zip(non_eeg_names, non_eeg_types) if t == "eeg"
		}
		if mapping_non_eeg_types:
			raw.set_channel_types(mapping_non_eeg_types)

	if mapping_eeg_names:
		# Set 1005 channels type to "eeg":
		raw.set_channel_types(
			{c: "eeg" for c in mapping_eeg_names}, on_unit_change="ignore"
		)
		# Fix capitalized EEG channel names:
		raw.rename_channels(mapping_eeg_names)

In [2]:
openneuro.download(dataset="ds005505", target_dir="SampleData", include=["sub-NDARAC904DMU"])


👋 Hello! This is openneuro-py 2024.2.0. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/hoechenberger/openneuro-py/issues

🌍 Preparing to download ds005505 …
Cannot determine local revision of the dataset, and the target directory is not empty. If the download fails, you may want to try again with a fresh (empty) target directory.


📁 Traversing directories for ds005505 : 0 entities [00:00, ? entities/s]

📥 Retrieving up to 76 files (5 concurrent downloads). 
✅ Finished downloading ds005505.
 
🧠 Please enjoy your brains.
 


dataset_description.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

participants.json:   0%|          | 0.00/2.72k [00:00<?, ?B/s]

...

sub-NDARAC904DMU_task-symbolSearch_eeg.json:   0%|          | 0.00/231 [00:00<?, ?B/s]

sub-NDARAC904DMU_task-surroundSupp_run-2_electrodes.tsv: 0.00B [00:00, ?B/s]

sub-NDARAC904DMU_task-symbolSearch_eeg.set:   0%|          | 0.00/39.4M [00:00<?, ?B/s]

sub-NDARAC904DMU_task-symbolSearch_events.tsv:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

sub-NDARAC904DMU_task-symbolSearch_electrodes.tsv:   0%|          | 0.00/4.11k [00:00<?, ?B/s]

In [33]:
TUH_patient = mne.io.read_raw_edf('SampleData/TUH/aaaaaaac_s001_t000.edf', preload=False)
TUH_patient.info["line_freq"] = 50
_rename_channels(TUH_patient)
montage = mne.channels.make_standard_montage("standard_1005")
TUH_patient.set_montage(montage, on_missing="ignore")
for channel in TUH_patient.info["chs"]:
	print(channel['loc'])

Extracting EDF parameters from /mnt/batch/tasks/shared/LS_root/mounts/clusters/mchen439-e8ads-v5/code/Users/mchen439/EEG-Foundation-Model/SampleData/TUH/aaaaaaac_s001_t000.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
[-0.03090259  0.11458518  0.02786657  0.          0.          0.                 nan         nan         nan         nan         nan         nan]
[0.0284095  0.11534631 0.02772126 0.         0.         0.                nan        nan        nan        nan        nan        nan]
[-0.05180904  0.0866879   0.07871409  0.          0.          0.                 nan         nan         nan         nan         nan         nan]
[0.05027428 0.08743839 0.07727065 0.         0.         0.                nan        nan        nan        nan        nan        nan]
[-0.06714872  0.02335823  0.10451068  0.          0.          0.                 nan         nan         nan         nan         nan         nan]
[0.06532888 0.0235731  0.1036924

/tmp/ipykernel_37774/1918410036.py:29: RuntimeWarning: The unit for channel(s) 30, EKG, LUC, PG1, PG2, PHOTIC PH, RLC, SP1, SP2, T1, T2 has changed from V to NA.
  raw.set_channel_types(mapping_non_eeg_types)


In [34]:
bids_path = BIDSPath(subject="aaaaaaac", task="Rest", root="SampleData/TUH-BIDS")
write_raw_bids(TUH_patient, bids_path, overwrite=True)

Extracting EDF parameters from /mnt/batch/tasks/shared/LS_root/mounts/clusters/mchen439-e8ads-v5/code/Users/mchen439/EEG-Foundation-Model/SampleData/TUH/aaaaaaac_s001_t000.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Writing 'SampleData/TUH-BIDS/README'...
Writing 'SampleData/TUH-BIDS/participants.tsv'...
Writing 'SampleData/TUH-BIDS/participants.json'...
Writing 'SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_space-CapTrak_electrodes.tsv'...
Writing 'SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_space-CapTrak_coordsystem.json'...
Writing 'SampleData/TUH-BIDS/dataset_description.json'...
Writing 'SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_task-Rest_eeg.json'...
Writing 'SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_task-Rest_channels.tsv'...
Copying data files to sub-aaaaaaac_task-Rest_eeg.edf


/tmp/ipykernel_37774/1519500572.py:2: RuntimeWarning: No events found or provided. Please add annotations to the raw data, or provide the events and event_id parameters. For resting state data, BIDS recommends naming the task using labels beginning with "rest".
  write_raw_bids(TUH_patient, bids_path, overwrite=True)


Writing 'SampleData/TUH-BIDS/sub-aaaaaaac/sub-aaaaaaac_scans.tsv'...
Wrote SampleData/TUH-BIDS/sub-aaaaaaac/sub-aaaaaaac_scans.tsv entry with eeg/sub-aaaaaaac_task-Rest_eeg.edf.


BIDSPath(
root: SampleData/TUH-BIDS
datatype: eeg
basename: sub-aaaaaaac_task-Rest_eeg.edf)

In [40]:
hbn_bids_path = BIDSPath(root="SampleData/HBN", session=None, datatype="eeg")
hbn_bids_path = hbn_bids_path.update(subject="NDARAC904DMU", task="RestingState", suffix="eeg")
HBN_patient = read_raw_bids(bids_path=hbn_bids_path, verbose=True)

/tmp/ipykernel_37774/2368780825.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  HBN_patient = read_raw_bids(bids_path=hbn_bids_path, verbose=True)


Reading events from SampleData/HBN/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-RestingState_events.tsv.
Reading channel info from SampleData/HBN/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-RestingState_channels.tsv.
Reading electrode coords from SampleData/HBN/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-RestingState_electrodes.tsv.


/tmp/ipykernel_37774/2368780825.py:3: RuntimeWarning: Unable to map the following column(s) to to MNE:
release_number: R1
ehq_total: 71.17
commercial_use: Yes
full_pheno: Yes
p_factor: -0.603
attention: -0.446
internalizing: 1.248
externalizing: 0.325
RestingState: available
DespicableMe: available
FunwithFractals: available
ThePresent: available
DiaryOfAWimpyKid: available
contrastChangeDetection_1: available
contrastChangeDetection_2: available
contrastChangeDetection_3: available
surroundSupp_1: available
surroundSupp_2: available
seqLearning6target: unavailable
seqLearning8target: available
symbolSearch: available
  HBN_patient = read_raw_bids(bids_path=hbn_bids_path, verbose=True)


In [41]:
tuh_bids_path = BIDSPath(root="SampleData/TUH-BIDS", session=None, datatype="eeg")
tuh_bids_path = tuh_bids_path.update(subject="aaaaaaac", task="Rest", suffix="eeg")
TUH_patient = read_raw_bids(bids_path=tuh_bids_path, verbose=True)

Extracting EDF parameters from /mnt/batch/tasks/shared/LS_root/mounts/clusters/mchen439-e8ads-v5/code/Users/mchen439/EEG-Foundation-Model/SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_task-Rest_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading channel info from SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_task-Rest_channels.tsv.
Reading electrode coords from SampleData/TUH-BIDS/sub-aaaaaaac/eeg/sub-aaaaaaac_space-CapTrak_electrodes.tsv.


/tmp/ipykernel_37774/3753720600.py:3: RuntimeWarning: Did not find any events.tsv associated with sub-aaaaaaac_task-Rest.

The search_str was "SampleData/TUH-BIDS/sub-aaaaaaac/**/eeg/sub-aaaaaaac*events.tsv"
  TUH_patient = read_raw_bids(bids_path=tuh_bids_path, verbose=True)
/tmp/ipykernel_37774/3753720600.py:3: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['PG1', 'PG2', 'EKG', 'SP2', 'SP1', 'RLC', 'LUC', '30', 'T1', 'T2', 'PHOTIC PH']
  TUH_patient = read_raw_bids(bids_path=tuh_bids_path, verbose=True)
/tmp/ipykernel_37774/3753720600.py:3: RuntimeWarning: Not setting positions of 11 misc channels found in montage:
['PG1', 'PG2', 'EKG', 'SP2', 'SP1', 'RLC', 'LUC', '30', 'T1', 'T2', 'PHOTIC PH']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  TUH_patient = read_raw_bids(bids_path=tuh_bids_path, verbose=True)


Not fully anonymizing info - keeping his_id, sex, and hand info


In [42]:
print(TUH_patient.info)
print(HBN_patient.info)

<Info | 12 non-empty values
 bads: []
 ch_names: Fp1, Fp2, F3, F4, C3, C4, A1, A2, P3, P4, O1, O2, F7, F8, T3, ...
 chs: 22 EEG, 11 misc
 custom_ref_applied: False
 description: Anonymized using a time shift to preserve age at acquisition
 dig: 25 items (3 Cardinal, 22 EEG)
 experimenter: mne_anonymize
 highpass: 0.0 Hz
 line_freq: 50.0
 lowpass: 125.0 Hz
 meas_date: 2002-01-01 00:00:00 UTC
 nchan: 33
 projs: []
 sfreq: 250.0 Hz
 subject_info: 3 items (dict)
>
<Info | 10 non-empty values
 bads: []
 ch_names: E1, E2, E3, E4, E5, E6, E7, E8, E9, E10, E11, E12, E13, E14, ...
 chs: 129 EEG
 custom_ref_applied: False
 dig: 129 items (129 EEG)
 highpass: 0.0 Hz
 line_freq: 60.0
 lowpass: 250.0 Hz
 meas_date: unspecified
 nchan: 129
 projs: []
 sfreq: 500.0 Hz
 subject_info: 2 items (dict)
>
